In [ ]:
%load_ext autoreload
%autoreload 2

from spatial_tcr.utils import setup_plotting, switch_cwd_to_root

switch_cwd_to_root()

import os

figure_dir = "figures/revision"
setup_plotting(figure_dir, display_dpi=300, save_dpi=300)

import numpy as np
import pandas as pd
import scanpy as sc
from tqdm.auto import tqdm

from spatial_tcr.clonal_expansion import (
    calc_empirical_p_values,
    identify_clonal_clusters_single_chain,
)
from spatial_tcr.permutation import permute_positions
from spatial_tcr.tcr import get_tcr_genes

In [ ]:
data_dir = "data/xenium/processed"
path = f"{data_dir}/08.2-kidney_tcr_clonal_clusters_peri_glom_annot.h5ad"
adata = sc.read_h5ad(path)

# remove control samples
adata = adata[adata.obs["condition"] == "ANCA"].copy()

In [ ]:
av_genes, bv_genes, dv_genes, gv_genes, tv_genes = get_tcr_genes(adata)

In [ ]:
ct_key = "cell_type_l2"
tcell_keys = [
    "CD4+",
    "Tregs",
    "MAIT",
    "CD8+",
    "NKT-like",
]


ct_key = "cell_type_l1.1"
tcell_keys = [
    "CD4+",
    "MAIT",
    "CD8+",
    "NKT-like",
]

max_dist = 100
min_cells = 4

identify_clonal_clusters_single_chain(
    adata,
    av_genes,
    layer="avbv_clean",
    ct_key=ct_key,
    tcell_keys=tcell_keys,
    clonal_cluster_key="av_cluster",
    max_dist=max_dist,
    min_cells=min_cells,
)

identify_clonal_clusters_single_chain(
    adata,
    bv_genes,
    layer="avbv_clean",
    ct_key=ct_key,
    tcell_keys=tcell_keys,
    clonal_cluster_key="bv_cluster",
    max_dist=max_dist,
    min_cells=min_cells,
)

In [ ]:
cc_key = "av_cluster_filtered"
# cc_key = "avbv_cluster"

for sample in adata.obs["sample"].unique()[1:2]:
    ad_sub = adata[adata.obs["sample"] == sample].copy()
    if ad_sub.obs[cc_key].notna().sum() == 0:
        continue
    sc.pl.spatial(
        ad_sub,
        color=cc_key,
        show=True,
        spot_size=20,
        cmap="nipy_spectral",
        title="TRAV single-chain clonal cluster",
    )


cc_key = "bv_cluster_filtered"
# cc_key = "avbv_cluster"

for sample in adata.obs["sample"].unique()[1:2]:
    ad_sub = adata[adata.obs["sample"] == sample].copy()
    if ad_sub.obs[cc_key].notna().sum() == 0:
        continue
    sc.pl.spatial(
        ad_sub,
        color=cc_key,
        show=True,
        spot_size=20,
        cmap="nipy_spectral",
        title="TRBV single-chain clonal cluster",
    )

cc_key = "avbv_cluster_filtered"

for sample in adata.obs["sample"].unique()[1:2]:
    ad_sub = adata[adata.obs["sample"] == sample].copy()
    if ad_sub.obs[cc_key].notna().sum() == 0:
        continue
    sc.pl.spatial(
        ad_sub,
        color=cc_key,
        show=True,
        spot_size=20,
        cmap="nipy_spectral",
        title="TRAV+BV clonal cluster",
    )

## Analyze clonal clusters

In [ ]:
ad_t = adata[adata.obs["cell_type_l1"] == "T"].copy()

In [ ]:
ct_key = "cell_type_l2"
clone_keys = ["av_cluster_filtered", "bv_cluster_filtered", "avbv_cluster_filtered"]


results = []
results_raw = pd.DataFrame(index=clone_keys + ["T"], columns=ad_t.obs[ct_key].unique())
background_comp = ad_t.obs[ct_key].value_counts(normalize=True)
for clone_key in clone_keys:
    mask_1 = ad_t.obs[clone_key] != "NA"
    mask_2 = ~ad_t.obs[clone_key].isna()
    ad_sub = ad_t[mask_1 & mask_2].copy()
    print(ad_sub.shape)
    clone_cluster_comp = ad_sub.obs[ct_key].value_counts(normalize=True)
    ccc_index = clone_cluster_comp.index
    result = (
        (
            100
            * (clone_cluster_comp - background_comp[ccc_index])
            / background_comp[ccc_index]
        )
        .to_frame()
        .reset_index()
    )
    results_raw.loc[clone_key, ccc_index] = clone_cluster_comp.values
    result["clone_key"] = clone_key
    results.append(result)
results_raw.loc["T", background_comp.index] = background_comp.values
results = pd.concat(results, axis=0)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import AutoMinorLocator

fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(data=results, x=ct_key, y="proportion", hue="clone_key", ax=ax)
ax.set_ylabel("Proportion change relative to background\nT subtype distribution [%]")
ax.set_title("T cell subtype composition of single chain clonal clusters")
ax.get_legend().set_title("")

ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
new_labels = ["AV", "BV", "AV+BV"]
for t, new_label in zip(ax.get_legend().texts, new_labels):
    t.set_text(new_label)
ax.set_xlabel("")
ax.yaxis.set_minor_locator(AutoMinorLocator(n=2))
ax.grid(which="both", axis="y", linestyle="--", linewidth=0.5)
# rotate x labels
ax.set_xticklabels(ax.get_xticklabels(), rotation=0, ha="center")
sns.despine(ax=ax)

In [ ]:
results_raw.sum(axis=1)

In [ ]:
fig, ax = plt.subplots(figsize=(2, 4))
results_raw.plot(kind="bar", stacked=True, ax=ax, rot=0)
ax.set_ylabel("T cell subtype proportion")
ax.set_title("Clonal cluster T subtype composition")
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
ax.set_xticklabels(["AV", "BV", "AV+BV", "all T"])
ax.set_ylim(0, 1)
sns.despine(ax=ax)

## Permutation testing

In [ ]:
ct_key = "cell_type_l1.1"
tcell_keys = [
    "CD4+",
    "MAIT",
    "CD8+",
    "NKT-like",
]

max_dist = 100
min_cells = 4
seed = 42

n_perms = 1000
layer = "avbv_clean"
# layer = "counts"

save_dir = "results/revision/clonal-expansion"
os.makedirs(save_dir, exist_ok=True)
save_path_av = (
    f"{save_dir}/single_chain_clonal_clusters_simulated_postionwise_av_{layer}.csv"
)
save_path_bv = (
    f"{save_dir}/single_chain_clonal_clusters_simulated_postionwise_bv_{layer}.csv"
)

In [ ]:
rng = np.random.RandomState(seed)
ad_tmp = adata[adata.obs[ct_key].isin(tcell_keys), av_genes + bv_genes].copy()

cluster_results_av = []
cluster_results_bv = []

for _ in tqdm(range(n_perms)):
    ad_perm = ad_tmp.copy()

    ad_perm = permute_positions(ad_perm, rng=rng, sample_key="cc")

    identify_clonal_clusters_single_chain(
        ad_perm,
        av_genes,
        layer="avbv_clean",
        ct_key=ct_key,
        tcell_keys=tcell_keys,
        clonal_cluster_key="av_cluster",
        max_dist=max_dist,
        min_cells=min_cells,
        verbose=False,
    )

    identify_clonal_clusters_single_chain(
        ad_perm,
        bv_genes,
        layer="avbv_clean",
        ct_key=ct_key,
        tcell_keys=tcell_keys,
        clonal_cluster_key="bv_cluster",
        max_dist=max_dist,
        min_cells=min_cells,
        verbose=False,
    )

    cluster_results_av.append(ad_perm.obs["av_cluster_filtered"])
    cluster_results_bv.append(ad_perm.obs["bv_cluster_filtered"])


cluster_results_av = pd.concat(cluster_results_av, axis=1)
cluster_results_bv = pd.concat(cluster_results_bv, axis=1)
cluster_results_av.to_csv(save_path_av)
cluster_results_bv.to_csv(save_path_bv)

In [ ]:
cluster_results_av = pd.read_csv(save_path_av, index_col=0)
cluster_results_bv = pd.read_csv(save_path_bv, index_col=0)

In [ ]:
def plot_permutation_test_results(
    counts_sim,
    counts_obs,
    p_values,
    title="clonal clusters in simulated data",
    save_path=None,
):
    sns.set_theme(style="ticks", context="paper")
    fig, ax = plt.subplots(figsize=(6, 4))
    hist = sns.histplot(counts_sim, ax=ax)
    ax.set_xlabel("number of clonal clusters")
    ax.set_ylabel("count")
    ax.set_title(title)

    # Get the bar coordinates
    # for p in hist.patches:
    #     print(f"Bar center: {p.get_x() + p.get_width() / 2:.2f}, height: {p.get_height()}")

    bar_centers = [p.get_x() + p.get_width() / 2 for p in hist.patches]
    print(bar_centers)

    # draw red vertical line for observed number of clonal clusters
    ax.axvline(counts_obs, color="red", linestyle="--")
    # add text for observed number of clonal clusters
    ymax = ax.get_ylim()[1]
    ax.text(
        counts_obs - 1,
        0.9 * ymax,
        f"observed number of clonal clusters: {counts_obs}\npermutation p-value: {np.round(p_values, 5)}",
        color="red",
        ha="right",
    )

    # split x axis into two parts
    sns.despine(ax=ax)

    if save_path is not None:
        plt.savefig(
            save_path,
            dpi=300,
            bbox_inches="tight",
            transparent=True,
        )

In [ ]:
n_perms = 1000
counts_sim = cluster_results_av.nunique().values
counts_obs = adata.obs["av_cluster_filtered"].nunique()
z_scores, p_values, q_values = calc_empirical_p_values(counts_obs, counts_sim, n_perms)
print(p_values)
print(q_values)
plot_permutation_test_results(
    counts_sim,
    counts_obs,
    p_values,
    title="AV clonal clusters in simulated data",
    save_path=os.path.join(figure_dir, "AV_clonal_cluster_perm_test.pdf"),
)

In [ ]:
n_perms = 1000
counts_sim = cluster_results_bv.nunique().values
counts_obs = adata.obs["bv_cluster_filtered"].nunique()
z_scores, p_values, q_values = calc_empirical_p_values(counts_obs, counts_sim, n_perms)
print(p_values)
print(q_values)
plot_permutation_test_results(
    counts_sim,
    counts_obs,
    p_values,
    title="BV clonal clusters in simulated data",
    save_path=os.path.join(figure_dir, "BV_clonal_cluster_perm_test.pdf"),
)